# **Semantic Web & Linked Open Data with Python**
## บทที่ 12 — Linked Open Data

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Rathachai/semantic-web/blob/main/code/ch12/ch12_notebook.ipynb)

> ## *Author* : [Rathachai CHAWUTHAI](https://rathachai.creatier.pro/) , Ph.D
> ### *Affiliation* : Computer Engineering, King Mongkut's Institute of Technology Ladkrabang (KMITL)
> #### *Updated Date* : 2026-08-23
---

<a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Creative Commons License" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/88x31.png" /></a><br />This work (course text) is licensed under a <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Creative Commons Attribution 4.0 International License</a>. Sample code cells are additionally released under the MIT License.

**References:**
*   https://5stardata.info/
*   https://www.w3.org/DesignIssues/LinkedData.html
*   https://www.wikidata.org/

📖 เอกสารประกอบบทนี้: [`12-linked-open-data/`](https://github.com/Rathachai/semantic-web/tree/main/12-linked-open-data)

---

### Setup

In [1]:
!pip install -q rdflib flask

In [2]:
import csv, io, re
from rdflib import Graph, Literal, Namespace, URIRef
from rdflib.namespace import OWL, RDF, RDFS, XSD

## จาก CSV (⭐⭐⭐) ไปสู่ RDF (⭐⭐⭐⭐)

ปัญหาคลาสสิกของ CSV: ชื่อเมืองสะกดต่างกัน — `'Bangkok'`, `' bangkok '`, `'BANGKOK'`

In [3]:
# ตัวอย่างข้อมูลดิบที่ไม่สะอาด -- สังเกตวิธีสะกดชื่อเมืองที่ต่างกัน
rows = [
    {"name_en": "Chulalongkorn University", "city": "Bangkok"},
    {"name_en": "Chiang Mai University", "city": " bangkok "},
    {"name_en": "Thammasat University", "city": "BANGKOK"},
]
cities_raw = {r["city"] for r in rows}
print("ค่าดิบที่ต่างกัน:", cities_raw)

ค่าดิบที่ต่างกัน: {' bangkok ', 'BANGKOK', 'Bangkok'}


## ทำความสะอาด + ออกแบบ URI

In [4]:
def slugify(text: str) -> str:
    s = text.lower()
    s = re.sub(r"[\'`]", "", s)
    s = re.sub(r"[^a-z0-9]+", "-", s)
    return s.strip("-")

BASE = "https://data.example.org/"
UNI = Namespace(BASE + "university/")
PLACE = Namespace(BASE + "place/")

g = Graph()
g.bind("uni", UNI); g.bind("place", PLACE)
for r in rows:
    city = r["city"].strip().title()
    u = UNI[slugify(r["name_en"])]
    p = PLACE[slugify(city)]
    g.add((u, RDF.type, URIRef(BASE + "vocab#University")))
    g.add((u, RDFS.label, Literal(r["name_en"], lang="en")))
    g.add((u, URIRef(BASE + "vocab#locatedIn"), p))
    g.add((p, RDFS.label, Literal(city, lang="en")))

cities_clean = {str(o) for _, _, o in g.triples((None, RDFS.label, None))
                if str(o) in {"Bangkok"}}
print(f"{len(g)} triples, เมืองที่ normalise แล้ว: {cities_clean}")

10 triples, เมืองที่ normalise แล้ว: {'Bangkok'}


## เชื่อมเข้า Wikidata (⭐⭐⭐⭐⭐)

In [5]:
WD = Namespace("http://www.wikidata.org/entity/")
bangkok_uri = PLACE[slugify("Bangkok")]
g.add((bangkok_uri, OWL.sameAs, WD.Q1861))

print(g.serialize(format="turtle"))

@prefix ns1: <https://data.example.org/vocab#> .
@prefix owl: <http://www.w3.org/2002/07/owl#> .
@prefix place: <https://data.example.org/place/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix uni: <https://data.example.org/university/> .

uni:chiang-mai-university a ns1:University ;
    rdfs:label "Chiang Mai University"@en ;
    ns1:locatedIn place:bangkok .

uni:chulalongkorn-university a ns1:University ;
    rdfs:label "Chulalongkorn University"@en ;
    ns1:locatedIn place:bangkok .

uni:thammasat-university a ns1:University ;
    rdfs:label "Thammasat University"@en ;
    ns1:locatedIn place:bangkok .

place:bangkok rdfs:label "Bangkok"@en ;
    owl:sameAs <http://www.wikidata.org/entity/Q1861> .




## Content Negotiation ด้วย Flask

URI ของ**สิ่งของ**ต้องตอบ **303 See Other** เสมอ ไม่ใช่ 200 — เพราะ 200 จะแปลว่า "URI นี้คือเอกสารที่ส่งกลับมา" ซึ่งผิด

In [6]:
from flask import Flask, Response, abort, redirect, request

app = Flask(__name__)
FORMATS = {
    "text/turtle": ("ttl", "turtle"),
    "application/ld+json": ("jsonld", "json-ld"),
    "text/html": ("html", None),
}

@app.route("/university/<ident>")
def thing(ident):
    ext, _ = FORMATS[request.accept_mimetypes.best_match(FORMATS, default="text/html")]
    return redirect(f"/doc/university/{ident}.{ext}", code=303)

@app.route("/doc/university/<ident>.<ext>")
def doc(ident, ext):
    uri = UNI[ident]
    sub = Graph()
    for t in g.triples((uri, None, None)):
        sub.add(t)
    if ext == "html":
        return Response(f"<h1>{ident}</h1>", mimetype="text/html")
    _, fmt = next(v for v in FORMATS.values() if v[0] == ext)
    return Response(sub.serialize(format=fmt), mimetype=request.args.get("accept", "text/turtle"))

client = app.test_client()
r = client.get("/university/chulalongkorn-university", headers={"Accept": "text/turtle"})
print("status:", r.status_code, "-> Location:", r.headers["Location"])

status: 303 -> Location: /doc/university/chulalongkorn-university.ttl


In [7]:
r2 = client.get(r.headers["Location"], headers={"Accept": "text/turtle"})
print("status:", r2.status_code)
print(r2.get_data(as_text=True))

status: 200
@prefix ns1: <https://data.example.org/vocab#> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .

<https://data.example.org/university/chulalongkorn-university> a ns1:University ;
    rdfs:label "Chulalongkorn University"@en ;
    ns1:locatedIn <https://data.example.org/place/bangkok> .




## VoID — บัตรประจำตัวของ dataset

In [8]:
VOID = Namespace("http://rdfs.org/ns/void#")
v = Graph()
v.bind("void", VOID)
dataset = URIRef(BASE + "dataset/universities")
v.add((dataset, RDF.type, VOID.Dataset))
v.add((dataset, VOID.triples, Literal(len(g), datatype=XSD.integer)))
v.add((dataset, VOID.uriSpace, Literal(BASE)))
print(v.serialize(format="turtle"))

@prefix void: <http://rdfs.org/ns/void#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://data.example.org/dataset/universities> a void:Dataset ;
    void:triples 11 ;
    void:uriSpace "https://data.example.org/" .




## 🎯 สรุป

- **CSV → RDF** แก้ปัญหาข้อมูลสะกดต่างกันได้ที่ต้นเหตุด้วย URI
- **`owl:sameAs`** เชื่อมเข้า Wikidata ทำให้ได้ label หลายภาษาและลิงก์ไปฐานข้อมูลอื่นฟรี
- URI ของสิ่งของต้องตอบ **303** เสมอ — ตรวจสอบได้ง่ายด้วย Flask test client
- **VoID** สรุปสถิติและจุดเข้าถึงของ dataset ให้เครื่องอ่านได้

ลองเพิ่มมหาวิทยาลัยอีกแห่ง แล้วดูว่า URI, RDF, และ Flask route ทำงานถูกต้องไหม!

---

## จบคอร์ส 🎉

ขอบคุณที่เรียนจนจบ! กลับไปทบทวนได้ที่ [https://github.com/Rathachai/semantic-web](https://github.com/Rathachai/semantic-web)

---
https://github.com/Rathachai/semantic-web

*Semantic Web & Linked Open Data with Python*